In [ ]:
import os
import pandas as pd
import numpy as np
import lightgbm as lgb
from sklearn.feature_extraction.text import TfidfVectorizer
import warnings

# Suppress warnings for a clean output console
warnings.filterwarnings('ignore')

def run_production_pipeline():
    print("--- Step 1: Locating and Loading Data ---")
    file_paths = {}
    for dirname, _, filenames in os.walk('/kaggle/input'):
        for filename in filenames:
            file_paths[filename] = os.path.join(dirname, filename)
            
    train_raw = pd.read_csv(file_paths["train.csv"])
    complaints = pd.read_csv(file_paths["chief_complaints.csv"])
    history = pd.read_csv(file_paths["patient_history.csv"])
    test_raw = pd.read_csv(file_paths["test.csv"])

    print("--- Step 2: Feature Engineering ---")
    df_train = train_raw.merge(complaints, on="patient_id", how="left").merge(history, on="patient_id", how="left")
    
    # Eradicate Leaky Data immediately
    leaky_cols = ['disposition', 'ed_los_hours']
    cols_to_drop = [c for c in leaky_cols if c in df_train.columns]
    df_train = df_train.drop(columns=cols_to_drop)

    # MNAR Imputation
    vitals = ['systolic_bp', 'diastolic_bp', 'heart_rate', 'respiratory_rate', 'temperature_c', 'spo2', 'pain_score']
    for v in vitals:
        if v in df_train.columns:
            df_train[f'{v}_is_missing'] = df_train[v].isnull().astype(int)
            df_train[v] = df_train[v].fillna(-999)

    # NLP Vectorization
    train_cc = df_train['chief_complaint_raw'].fillna('none')
    tfidf = TfidfVectorizer(max_features=50, stop_words='english', ngram_range=(1,2))
    train_text_features = tfidf.fit_transform(train_cc).toarray()
    text_cols = [f'tfidf_{w}' for w in tfidf.get_feature_names_out()]
    
    text_df = pd.DataFrame(train_text_features, columns=text_cols)
    df_train = pd.concat([df_train.reset_index(drop=True), text_df.reset_index(drop=True)], axis=1)
    df_train = df_train.drop(columns=['chief_complaint_raw'])

    # Categorical Encoding
    categorical_cols = df_train.select_dtypes(include=['object', 'category']).columns
    for col in categorical_cols:
        if col != 'patient_id':
            df_train[col] = df_train[col].astype('category').cat.codes

    print("--- Step 3: Leak-Free Model Training ---")
    X_train = df_train.drop(columns=['patient_id', 'triage_acuity'])
    y_train = df_train['triage_acuity'] - 1  # Shift 1-5 to 0-4 for LightGBM
    
    params = {
        'objective': 'multiclass', 'num_class': 5, 'metric': 'multi_logloss',
        'boosting_type': 'gbdt', 'learning_rate': 0.05, 'num_leaves': 45,
        'class_weight': 'balanced', 'random_state': 42, 'verbose': -1
    }
    model = lgb.LGBMClassifier(**params)
    model.fit(X_train, y_train)

    print("--- Step 4: Processing Test Data & Predicting ---")
    df_test = test_raw.merge(complaints, on="patient_id", how="left").merge(history, on="patient_id", how="left")
    
    for v in vitals:
        if v in df_test.columns:
            df_test[f'{v}_is_missing'] = df_test[v].isnull().astype(int)
            df_test[v] = df_test[v].fillna(-999)
            
    test_cc = df_test['chief_complaint_raw'].fillna('none')
    test_text_features = tfidf.transform(test_cc).toarray()
    test_text_df = pd.DataFrame(test_text_features, columns=text_cols)
    
    df_test = pd.concat([df_test.reset_index(drop=True), test_text_df.reset_index(drop=True)], axis=1)
    df_test = df_test.drop(columns=['chief_complaint_raw'])
    
    # Safe Categorical Encoding for test set
    for col in categorical_cols:
        if col != 'patient_id' and col in df_test.columns:
            df_test[col] = df_test[col].astype('category').cat.codes
            
    X_test = df_test.drop(columns=['patient_id'])
    X_test = X_test.reindex(columns=X_train.columns, fill_value=0)
    
    test_predictions = model.predict(X_test) + 1 
    
    submission = pd.DataFrame({'patient_id': df_test['patient_id'], 'triage_acuity': test_predictions})
    submission.to_csv('submission.csv', index=False)
    print(f"Success! Saved submission.csv with {len(submission)} predictions.")

if __name__ == "__main__":
    run_production_pipeline()

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os
import warnings
warnings.filterwarnings('ignore')

print("--- Generating Kaggle Writeup Images ---")

# 1. Load the training data dynamically
file_paths = {}
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        file_paths[filename] = os.path.join(dirname, filename)
df = pd.read_csv(file_paths["train.csv"])

# 2. Generate the 560x280 Thumbnail (ESI Distribution)
# figsize 5.6 x 2.8 at 100 DPI = exactly 560x280 pixels
fig_thumb, ax_thumb = plt.subplots(figsize=(5.6, 2.8), dpi=100) 
sns.countplot(data=df, x='triage_acuity', palette="Reds_r", ax=ax_thumb)
ax_thumb.set_title("Target Distribution: Triage Acuity", fontsize=12, fontweight='bold')
ax_thumb.set_xlabel("ESI Level (1 = Most Urgent)")
ax_thumb.set_ylabel("Patient Count")
plt.tight_layout()

thumb_path = 'thumbnail_560x280.png'
fig_thumb.savefig(thumb_path)
print(f"✅ Success! Saved {thumb_path} (Use for Card and Thumbnail)")

# 3. Generate the MNAR Heatmap for the Media Gallery
vitals = ['systolic_bp', 'diastolic_bp', 'heart_rate', 'respiratory_rate', 'temperature_c', 'spo2']
missing_rates = df.groupby('triage_acuity')[vitals].apply(lambda x: x.isnull().mean() * 100)

fig_gallery, ax_gallery = plt.subplots(figsize=(10, 6))
sns.heatmap(missing_rates, annot=True, fmt=".1f", cmap="YlOrRd", cbar_kws={'label': '% Missing'})
ax_gallery.set_title("Missing Not At Random (MNAR) Vital Signs by Acuity", fontsize=14, fontweight='bold')
ax_gallery.set_ylabel("ESI Level")
ax_gallery.set_xlabel("Vital Signs")
plt.tight_layout()

gallery_path = 'gallery_mnar_heatmap.png'
fig_gallery.savefig(gallery_path)
print(f"✅ Success! Saved {gallery_path} (Use for Media Gallery)")